# 7-to-1 |Y⟩ State Distillation (Steane Code)

Reference: `processing/Steane_distillation.png` Figure 46(c)

**Circuit structure:**
- W1, W2, W3: injected |Y⟩ (noisy)
- W4: fault-tolerant |+⟩
- A: ancilla |Y⟩ (reused per π/4 rotation)
- 4 sequential Z product measurements
- End: MX on W1-W3 (post-select), S†+MX on W4 (output)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.qec_code.surface_code.unrotated import (
    UnrotatedSurfaceCode,
    UnrotatedSurfaceCodeExtractionBlock,
    UnrotatedMultiPatchCoupler,
    UnrotatedSurfaceCodeLogicalOpSet,
)
from src.ir.qec_system import QECSystem
from src.ir.builder import CircuitBuilder
from src.ir.tracker import SyndromeTracker
import numpy as np

## Step 1: Single π/4 Rotation Test

Layout from `processing/Distillation layout.jpeg`:
```
W1(-2,0)    W3(10,0)
W2(-2,8)    W4(10,8)   ← IDLE
W5(-2,16)              ← Magic state
```

Protocol:
1. W1,W2,W3,W4 init Z (working qubits)
2. W5 init |+⟩ then transversal S → |Y⟩ (magic state)
3. ZZZZ measurement on {W1, W2, W3, W5} (W4 idle)
4. MX on W5, MZ on W1-W4, MX on coupler data
5. Expected: 4 logical observables (Z on W1-W4 preserved)

In [ ]:
d = 3
rounds = d
coupler_name = 'zzzz'

# --- Patch Setup ---
patch_layout = {
    'W1': (-2, 0),    # left-top
    'W3': (10, 0),    # right-top
    'W2': (-2, 8),    # left-mid
    'W4': (10, 8),    # right-mid (IDLE)
    'W5': (-2, 16),   # left-bottom (magic state)
}

system = QECSystem()
for name, offset in patch_layout.items():
    p = UnrotatedSurfaceCode(distance=d)
    p.transpose_coords()
    system.add_patch(p, name=name, offset=offset)

# Register coupler: ZZZZ on W1, W2, W3, W5 (W4 idle)
system.register_coupler(
    UnrotatedMultiPatchCoupler(),
    patch_names=['W1', 'W2', 'W3', 'W5'],
    name=coupler_name,
    path_axis='vertical',
    center_axis=6.0,
)

cp = system.coupler_patches[coupler_name]
num_qubits = system.num_qubits
op_set = UnrotatedSurfaceCodeLogicalOpSet(UnrotatedSurfaceCodeExtractionBlock)

print("Patch bounds:")
for name in patch_layout:
    role = 'IDLE' if name == 'W4' else ('MAGIC' if name == 'W5' else 'WORK')
    print(f"  {name} ({role}): {system.patches[name][0]._get_bounds()}")
print(f"\nCoupler: {len(cp.data_indices)} data, {len(cp.syndrome_indices)} syn")
print(f"System: {num_qubits} qubits, {system.num_logicals} logicals")

# --- Build Circuit ---
tracker = SyndromeTracker(num_qubits=num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
builder.write_coordinates()

# 1. Init W1,W2,W3,W4 in Z
working_data = {q: 'Z' for q in system.data_indices
                if system.index_to_owner_map[q] in ('W1','W2','W3','W4')}
builder.initialize(init_dict=working_data, n=num_qubits)

# 2. Init W5 in X (|+⟩)
w5_data = {q: 'X' for q in system.data_indices
           if system.index_to_owner_map[q] == 'W5'}
builder.initialize(init_dict=w5_data, n=num_qubits)

# 3. Apply transversal S on W5 → |+⟩ → |Y⟩
w5_patch = system.patches['W5'][0]
op_set.fold_transversal_s(builder, w5_patch)

# 4. SE rounds (stabilize all patches)
se = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

# 5. Activate coupler + init coupler data in X
builder.activate_coupler(coupler_name)
coupler_data_global = [system.local_to_global_map[coupler_name][q] for q in cp.data_indices]
coupler_init = {q: 'X' for q in coupler_data_global}
builder.initialize(init_dict=coupler_init, n=num_qubits)

# 6. SE with coupler active
se2 = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se2.circuit, rounds=rounds)

# 7. Final readout
# MZ on W1,W2,W3,W4; MX on W5; MX on coupler data
measure_dict = {}
for q in system.data_indices:
    owner = system.index_to_owner_map.get(q)
    if owner in ('W1', 'W2', 'W3', 'W4'):
        measure_dict[q] = 'Z'
    elif owner == 'W5':
        measure_dict[q] = 'X'
measure_dict.update(coupler_init)  # coupler data in X
builder.apply_data_readout(final_measurements=measure_dict)

circuit = builder.circuit
print(f"\nCircuit: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")

circuit.diagram("detslice-with-ops-svg")